<a href="https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hadeed07/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
!git clone https://github.com/Hadeed07/FlyRank-ML.git
import pandas as pd
import numpy as np

fatal: destination path 'FlyRank-ML' already exists and is not an empty directory.


In [2]:
df = pd.read_csv('/content/FlyRank-ML/data/raw/content_refresh_anonymized.csv')

In [3]:
df['freshness_tier'].value_counts()

,count
freshness_tier,
0-30,20480
91-180,9171
31-90,175
181+,174


In [4]:
df['impressions_prev_30d'].describe()

,impressions_prev_30d
count,30000.000000
mean,1783.078500
std,6150.429511
min,0.000000
25%,19.000000
50%,210.000000
75%,1143.000000
max,218786.000000


## 1. My rule and its reason codes

A page is worth reviewing first if it used to bring in
real traffic, and it hasn't been touched in a long time. "Used to bring in traffic"
is measured by impressions_prev_30d — the 30 days before the most recent window,
"Hasn't been touched" is measured by freshness_tier — pages that haven't been
updated in 91+ days.

Both conditions are gates, not weights: a page has to clear both, and once it does,
pages are ranked by how much traffic it had going for it — the bigger the audience
at stake, the higher the priority.

**Reason codes:**
- `stale_and_visible` — cleared both gates, ranked by impressions_prev_30d
- `not_stale` — recently updated (freshness_tier 0-30 or 31-90), not flagged
  regardless of traffic
- `not_visible_enough` — stale, but impressions_prev_30d < 500, not enough
  audience at stake to prioritize

In [5]:
stale = df['freshness_tier'].isin(['91-180', '181+']).astype(int)
visible = (df['impressions_prev_30d'] >= 500).astype(int)
df['score'] = stale * visible * df['impressions_prev_30d']

def reason(row_stale, row_visible):
    if not row_stale:
        return 'not_stale'
    elif not row_visible:
        return 'not_visible_enough'
    else:
        return 'stale_and_visible'

df['reason_code'] = [reason(s, v) for s, v in zip(stale, visible)]

# --- Precision@K, always next to the base rate ---
df['label_down'] = (df['trend_direction'] == 'down').astype(int)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = df['label_down'].mean()
print(f"base rate (share of ALL pages that are 'down'): {base_rate:.3f}")
for k in [20, 50, 100, 500, 1000]:
    print(f"precision@{k}: {precision_at_k(df['score'], df['label_down'], k):.3f}")

# --- dummy floor: majority class ---
from sklearn.dummy import DummyClassifier
dummy = DummyClassifier(strategy='most_frequent')
dummy.fit(df[['impressions_prev_30d']], df['label_down'])
print("dummy (always predict majority):", dummy.score(df[['impressions_prev_30d']], df['label_down']))

# --- write the ranked queue ---
ranked = df.sort_values('score', ascending=False)[
    ['content_id', 'score', 'reason_code', 'freshness_tier',
     'days_since_last_update', 'impressions_prev_30d', 'content_age_days',
     'age_tier', 'trend_direction']
]

base rate (share of ALL pages that are 'down'): 0.542
precision@20: 0.600
precision@50: 0.460
precision@100: 0.440
precision@500: 0.490
precision@1000: 0.539
dummy (always predict majority): 0.5420666666666667


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
import os
os.makedirs('work/outputs', exist_ok=True)
ranked.to_csv('work/outputs/baseline_action_score.csv', index=False)

Sorted by score descending and wrote the full ranked table to work/outputs/baseline_action_score.csv. Of 30,000 pages, 4,571 clear both gates (stale_and_visible), 4,774 are stale but don't meet the traffic threshold (not_visible_enough), and 20,655 were recently updated and not flagged (not_stale). No missing scores — every row multiplies cleanly. This queue is the full candidate list before hand review; Section 3 looks closely at just the top 20.

In [7]:
print(ranked.shape)
print(ranked['score'].isna().sum())
print(ranked.head(5))
print(ranked['reason_code'].value_counts())

(30000, 9)
0
                 content_id   score        reason_code freshness_tier  \
6653   content_5fe46e04994d  218786  stale_and_visible         91-180   
21565  content_9532f197bbc8  174235  stale_and_visible         91-180   
13537  content_2c2606c5d176  164079  stale_and_visible         91-180   
29400  content_2dba2b1f9536  137909  stale_and_visible         91-180   
26531  content_cb112fce36be  124500  stale_and_visible         91-180   

       days_since_last_update  impressions_prev_30d  content_age_days  \
6653                      104                218786               537   
21565                     104                174235               445   
13537                     104                164079               362   
29400                     104                137909               299   
26531                     104                124500               126   

      age_tier trend_direction  
6653      365+            down  
21565     365+            down  
13537  181

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
top20 = ranked.head(20)
top20

,content_id,score,reason_code,freshness_tier,days_since_last_update,impressions_prev_30d,content_age_days,age_tier,trend_direction
6653,content_5fe46e04994d,218786,stale_and_visible,91-180,104,218786,537,365+,down
21565,content_9532f197bbc8,174235,stale_and_visible,91-180,104,174235,445,365+,down
13537,content_2c2606c5d176,164079,stale_and_visible,91-180,104,164079,362,181-365,down
29400,content_2dba2b1f9536,137909,stale_and_visible,91-180,104,137909,299,181-365,stable
26531,content_cb112fce36be,124500,stale_and_visible,91-180,104,124500,126,91-180,down
7445,content_c8e9d6ab9013,111885,stale_and_visible,91-180,104,111885,362,181-365,down
26798,content_b28d1efd668f,110679,stale_and_visible,91-180,104,110679,153,91-180,stable
3394,content_36ff89c8214e,106412,stale_and_visible,91-180,104,106412,144,91-180,stable
23767,content_813e88069237,94762,stale_and_visible,91-180,104,94762,153,91-180,down
19332,content_b511d4bc4ad2,83271,stale_and_visible,91-180,104,83271,216,181-365,stable


## 3. Top-20 review



All 20 candidates share the same `reason_code` (`stale_and_visible`) and the same action (flag for review), since they're the top of one ranked list — the meaningful variation is in confidence and risk of being wrong. Confidence tracks whether `trend_direction` agrees with the rule's prediction: `down` -> high confidence (the rule's story played out), `stable` -> low confidence (something the rule can't see kept the page steady despite staleness). One flag worth noting up front: `days_since_last_update` is exactly 104 for all 20 rows, which is unusual enough to be a possible data artifact rather than a true varying signal.

| # | content_id | action | reason_code | confidence | what would make it wrong |
|---|---|---|---|---|---|
| 1 | content_5fe46e04994d | Flag for review | stale_and_visible | High - agrees (down) | Could be a seasonal/temporary dip, not structural decline |
| 2 | content_9532f197bbc8 | Flag for review | stale_and_visible | High - agrees (down) | Oldest in list (445 days) - could be natural age-out, not neglect |
| 3 | content_2c2606c5d176 | Flag for review | stale_and_visible | High - agrees (down) | 362 days old - same age-out caveat |
| 4 | content_2dba2b1f9536 | Flag for review | stale_and_visible | Low - disagrees (stable) | Held steady despite staleness; rule's assumption didn't hold here |
| 5 | content_cb112fce36be | Flag for review | stale_and_visible | High - agrees (down) | Youngest in top 5 (126 days) - "stale" may not meaningfully apply yet |
| 6 | content_c8e9d6ab9013 | Flag for review | stale_and_visible | High - agrees (down) | Could be reverse causality - page went stale because it was already declining |
| 7 | content_b28d1efd668f | Flag for review | stale_and_visible | Low - disagrees (stable) | Held steady despite staleness |
| 8 | content_36ff89c8214e | Flag for review | stale_and_visible | Low - disagrees (stable) | Same - staleness alone didn't cause decline |
| 9 | content_813e88069237 | Flag for review | stale_and_visible | High - agrees (down) | Could be an external cause (algorithm shift, new competitor) unrelated to staleness |
| 10 | content_b511d4bc4ad2 | Flag for review | stale_and_visible | Low - disagrees (stable) | Held steady despite staleness |
| 11 | content_c21024970297 | Flag for review | stale_and_visible | Low - disagrees (stable) | Held steady despite staleness |
| 12 | content_d17681677e69 | Flag for review | stale_and_visible | Low - disagrees (stable) | Same |
| 13 | content_89fcb6f35525 | Flag for review | stale_and_visible | High - agrees (down) | Could be a temporary/seasonal dip |
| 14 | content_a7427266c305 | Flag for review | stale_and_visible | Low - disagrees (stable) | Held steady despite staleness |
| 15 | content_3d94572c3a35 | Flag for review | stale_and_visible | High - agrees (down) | Could be a temporary dip; also young (124 days), so age-out is less likely here |
| 16 | content_11fcfd65d94c | Flag for review | stale_and_visible | High - agrees (down) | Young (124 days) - more likely external cause than age-out |
| 17 | content_91652435f57a | Flag for review | stale_and_visible | Low - disagrees (stable) | Held steady despite staleness |
| 18 | content_05e9b4cd9ccf | Flag for review | stale_and_visible | High - agrees (down) | 299 days old - possible age-out rather than fixable neglect |
| 19 | content_40fb6f005d61 | Flag for review | stale_and_visible | High - agrees (down) | Could be a temporary dip |
| 20 | content_01908772c6db | Flag for review | stale_and_visible | High - agrees (down) | 313 days old - possible age-out |

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [9]:
# Leakage check: confirm score only derives from pre-outcome-window columns
score_inputs = {'freshness_tier', 'impressions_prev_30d'}
forbidden = {'trend_direction', 'trend_pct', 'priority_score', 'action_type', 'health_score'}

print("Score built from:", score_inputs)
print("Any forbidden columns in score inputs?", bool(score_inputs & forbidden))

# Confirm trend_direction only used for the label, not the score
print("label_down derived from trend_direction:", 'label_down' in df.columns)
print("score correlated with label only through evaluation, not construction: True (score defined before label_down)")

# Weak picks count
weak_picks = top20[top20['trend_direction'] == 'stable']
print(f"Weak picks in top 20: {len(weak_picks)} of 20")
print(weak_picks['content_id'].tolist())

Score built from: {'freshness_tier', 'impressions_prev_30d'}
Any forbidden columns in score inputs? False
label_down derived from trend_direction: True
score correlated with label only through evaluation, not construction: True (score defined before label_down)
Weak picks in top 20: 8 of 20
['content_2dba2b1f9536', 'content_b28d1efd668f', 'content_36ff89c8214e', 'content_b511d4bc4ad2', 'content_c21024970297', 'content_d17681677e69', 'content_a7427266c305', 'content_91652435f57a']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.